In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

In [1]:
import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import autocast
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import f1_score, accuracy_score, classification_report

import transformers.utils.hub as _hub
_hub.list_repo_templates = lambda *a, **kw: []

from transformers import BlipProcessor, BlipForImageTextRetrieval

D:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CHECKPOINT   = r"C:\Users\Youssef Elghandour\Downloads\blip_itm_finetuned_best.pt"

IMAGES_ROOT  = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'origin')
VAL_LABELS   = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'merged_balanced', 'val.json')
VAL_META     = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'metadata', 'val.json')
LOCAL_MODEL  = _os.path.join(str(_cfg.ROOT), 'models', 'blip_itm_finetuned')

BATCH_SIZE   = 32
NUM_WORKERS  = 0

In [3]:
def build_image_path(raw_path):
    stripped = raw_path.removeprefix("visual_news/")
    return os.path.join(IMAGES_ROOT, stripped)


def load_val_split():
    with open(VAL_LABELS) as f:
        annotations = json.load(f)["annotations"]
    with open(VAL_META) as f:
        metadata = json.load(f)

    samples, missing_meta, missing_file = [], 0, 0
    for ann in annotations:
        key = str(ann["image_id"])
        if key not in metadata:
            missing_meta += 1
            continue
        meta     = metadata[key]
        img_path = build_image_path(meta["image_path"])
        if not os.path.exists(img_path):
            missing_file += 1
            continue
        samples.append({
            "image_path": img_path,
            "caption":    meta.get("caption", ""),
            "label":      int(ann["falsified"]),
        })

    print(f"[VAL] loaded={len(samples)} | missing_meta={missing_meta} | missing_file={missing_file}")
    return samples


class ITMDataset(Dataset):
    def __init__(self, samples, processor, max_text_len=128):
        self.samples      = samples
        self.processor    = processor
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            image = Image.open(s["image_path"]).convert("RGB")
        except (UnidentifiedImageError, OSError):
            image = Image.new("RGB", (384, 384))

        encoding = self.processor(
            images=image,
            text=s["caption"],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(s["label"], dtype=torch.long)
        return item

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

processor = BlipProcessor.from_pretrained(LOCAL_MODEL, local_files_only=True)
model     = BlipForImageTextRetrieval.from_pretrained(LOCAL_MODEL, local_files_only=True)

state_dict = torch.load(CHECKPOINT, map_location="cpu")
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("Checkpoint loaded.")

Device: cuda


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 472/472 [00:00<00:00, 6774.64it/s]

Checkpoint loaded.


In [5]:
val_samples = load_val_split()
val_ds      = ITMDataset(val_samples, processor)
val_loader  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"]

        with autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_itm_head=True,
            )

        preds = outputs.itm_score.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="binary", zero_division=0)

print(f"\nVal Accuracy : {acc:.4f}")
print(f"Val F1 (bin) : {f1:.4f}")
print()
print(classification_report(all_labels, all_preds,
                             target_names=["Real", "Fake"], zero_division=0))

[VAL] loaded=7024 | missing_meta=0 | missing_file=0


C:\Users\Youssef Elghandour\AppData\Local\Temp\ipykernel_18600\1474474665.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Val Accuracy : 0.5083
Val F1 (bin) : 0.5956

              precision    recall  f1-score   support

        Real       0.51      0.29      0.37      3512
        Fake       0.51      0.72      0.60      3512

    accuracy                           0.51      7024
   macro avg       0.51      0.51      0.48      7024
weighted avg       0.51      0.51      0.48      7024

